In [2]:
import pandas as pd
import sqlite3

# sqlite3 is built into Python — no install needed
# It creates a lightweight database stored as a single file

# Connect to or create a database file
conn = sqlite3.connect('data/kenya_economics.db')

#Load your master CSV into the database as a table called 'macro'
df = pd.read_csv('data/cleaned/master_annual.csv')
df.to_sql('macro', conn, if_exists='replace', index=False)

# Load the wages detail table — this has industry-level data
# which the master table doesn't have (master only has averages)
wages = pd.read_csv('data/cleaned/knbs_wages.csv')
wages.to_sql('wages', conn, if_exists='replace', index=False)

# Load CPI categories detail table
cpi = pd.read_csv('data/cleaned/knbs_cpi_categories.csv')
cpi.to_sql('cpi_categories', conn, if_exists='replace', index=False)

print("Tables created:")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
for row in cursor.fetchall():
    print(f" {row[0]}")

Tables created:
 macro
 wages
 cpi_categories


In [3]:
# How to run SQL in Python:  
# This is the pattern we use for every query today
# pd.read_sql() runs a SQL query and returns the result as a DataFrame
result = pd.read_sql("""
                     SELECT *
                     FROM macro
                     LIMIT 3
                     """, conn)
print(result)

   year  inflation_pct  gdp_per_capita_usd  gdp_per_capita_kes  \
0  2018       4.689806         1836.452755         186035.5547   
1  2019       5.239638         1960.408089         199944.5656   
2  2020       5.405162         1927.664590         205201.3992   

   gdp_growth_pct  cumulative_price_index  gdp_per_capita_kes_real  \
0        5.647946              100.000000              186035.5547   
1        5.114159              105.239638              189989.7885   
2       -0.272766              110.928011              184986.0981   

   nominal_growth_vs_2018_pct  real_growth_vs_2018_pct  \
0                    0.000000                 0.000000   
1                    7.476534                 2.125526   
2                   10.302248                -0.564116   

   purchasing_power_of_1000kes  ...  diesel_kes_litre  kerosene_kes_litre  \
0                  1000.000000  ...               NaN                 NaN   
1                   950.212315  ...            102.92              

*The triple quotes """...""" let you write multi-line SQL cleanly. Everything inside is pure SQL — Python just sends it to the database and hands back the result.*

**Query 1 — Basic SELECT with calculated columns**  
Concept: SELECT, column aliases, ROUND()
This is the foundation of all SQL. You tell the database which columns you want, from which table, and optionally what to call them. You can also create new columns by doing arithmetic directly in the SELECT statement.

In [ ]:
# ROUND(value, decimal_places) rounds a number
# column AS new_name gives a column a readable alias
# We're asking: show me the gap between nominal and real growth each year

q1 = pd.read_sql("""
    SELECT
        year,
        ROUND(nominal_growth_vs_2018_pct, 1)  AS nominal_growth_pct,
        ROUND(real_growth_vs_2018_pct, 1)     AS real_growth_pct,
        ROUND(nominal_growth_vs_2018_pct 
              - real_growth_vs_2018_pct, 1)   AS inflation_erosion_gap_pct,
        ROUND(purchasing_power_of_1000kes, 0) AS purchasing_power_kes
    FROM macro
    ORDER BY year
""", conn)

print("Query 1 — Nominal vs Real Growth Gap:")
print(q1.to_string(index=False))

# What every clause does:
#SELECT — choose which columns to return
#FROM macro — which table to read from
#ROUND(x, 1) — round to 1 decimal place
#AS — rename the column in the output
#ORDER BY year — sort results by year ascending

Query 1 — Nominal vs Real Growth Gap:
 year  nominal_growth_pct  real_growth_pct  inflation_erosion_gap_pct  purchasing_power_kes
 2018                 0.0              0.0                        0.0                1000.0
 2019                 7.5              2.1                        5.4                 950.0
 2020                10.3             -0.6                       10.9                 901.0
 2021                21.5              3.2                       18.3                 850.0
 2022                33.7              5.5                       28.2                 789.0
 2023                46.0              7.0                       39.0                 733.0
 2024                54.5              8.4                       46.1                 701.0


**Query 2 — WHERE and CASE**  
Concept: filtering rows and categorising values
WHERE filters which rows come back — like a Python boolean mask but written in plain English. CASE is SQL's version of an if/elif/else — it lets you create a new column whose value depends on a condition.

In [5]:
# WHERE filters rows — only return years where inflation was above 6%
# CASE creates a new column by testing conditions top to bottom
# WHEN condition THEN value — like elif
# ELSE catches everything that didn't match
# END closes the CASE statement

q2 = pd.read_sql("""
    SELECT
        year,
        ROUND(inflation_pct, 2)          AS inflation_pct,
        ROUND(gdp_growth_pct, 2)         AS gdp_growth_pct,
        ROUND(govt_debt_pct_gdp, 1)      AS govt_debt_pct_gdp,
        CASE
            WHEN inflation_pct >= 7.0 THEN 'High pressure'
            WHEN inflation_pct >= 5.5 THEN 'Elevated'
            WHEN inflation_pct >= 4.0 THEN 'Moderate'
            ELSE 'Low'
        END                              AS inflation_regime,
        CASE
            WHEN gdp_growth_pct < 0     THEN 'Contraction'
            WHEN gdp_growth_pct < 3.0   THEN 'Weak growth'
            WHEN gdp_growth_pct < 6.0   THEN 'Moderate growth'
            ELSE                             'Strong growth'
        END                             AS growth_regime
    FROM macro
    ORDER BY year
""", conn)

print("Query 2 — Inflation and Growth Regimes:")
print(q2.to_string(index=False))

Query 2 — Inflation and Growth Regimes:
 year  inflation_pct  gdp_growth_pct  govt_debt_pct_gdp inflation_regime   growth_regime
 2018           4.69            5.65               56.4         Moderate Moderate growth
 2019           5.24            5.11               59.1         Moderate Moderate growth
 2020           5.41           -0.27               68.0         Moderate     Contraction
 2021           6.11            7.59               68.2         Elevated   Strong growth
 2022           7.66            4.86               67.8    High pressure Moderate growth
 2023           7.67            5.72               73.1    High pressure Moderate growth
 2024           4.49            4.72               69.9         Moderate Moderate growth


## Query 2 — What the Output Tells Us

### What we built
Two classification columns using CASE — one for inflation severity, 
one for GDP growth strength. Every year gets labelled based on where 
its numbers fall in a defined range. This turns raw figures into 
categories a non-technical reader can immediately understand.

### Reading the table as a timeline

**2018–2019 — The stable baseline**
Moderate inflation, moderate growth, debt at 56–59% of GDP. 
Kenya's pre-shock normal. Not spectacular but manageable. 
Households were not thriving but they were not losing ground badly either.

**2020 — The COVID year**
The only contraction in the dataset. GDP shrank -0.27%. But inflation 
was still 5.4% — classified Moderate. This is the dangerous combination: 
the economy shrank AND prices still rose. People lost income AND lost 
purchasing power simultaneously. That is why 2020 real wages barely 
moved despite nominal increases.

**2021 — The rebound**
Strong growth at 7.59% as the economy reopened. But inflation jumped 
to 6.1% — Elevated. The recovery was real but inflation was already 
accelerating. Debt hit 68.2% of GDP as government borrowed heavily 
through COVID.

**2022–2023 — The pressure years**
The hardest period in the dataset. High pressure inflation (7.66%, 7.67%) 
combined with only Moderate growth. Not technically stagflation — growth 
was positive — but for ordinary households it felt like it. Wages were 
not keeping up, fuel was surging, food was expensive, and government debt 
peaked at 73.1% of GDP in 2023.

**2024 — The relief**
Inflation drops back to Moderate at 4.49%. The pressure eases. But 
purchasing power does not recover — the damage from 2020–2023 is already 
baked in. The shilling that was lost does not come back just because 
inflation slowed down.

### The one thing this table proves
Kenya never had a single year of truly low inflation AND strong growth 
simultaneously. Every strong growth year had elevated or rising inflation. 
Every low inflation year had weak or negative growth.

That tradeoff is your macro policy argument — and it is sitting right 
there in SQL.

**Query 3 — GROUP BY and aggregate functions**  
Concept: summarising groups of data
GROUP BY is one of the most important SQL concepts. Instead of returning one row per record, it collapses rows into groups and lets you calculate summaries — averages, totals, maximums — for each group.
Think of it like groupby() in pandas, because that's exactly where pandas got the idea.

In [6]:
# We're going to query the WAGES table now — not macro
# wages has 410 rows: one per industry, sector, year, and wage type
# GROUP BY sector, wage_type collapses all industries into two groups
# AVG() calculates the mean, MIN() the lowest, MAX() the highest

q3 = pd.read_sql("""
    SELECT
        sector,
        wage_type,
        COUNT(*)                              AS total_records,
        ROUND(AVG(monthly_wage_kes), 0)       AS avg_monthly_wage,
        ROUND(MIN(monthly_wage_kes), 0)       AS min_monthly_wage,
        ROUND(MAX(monthly_wage_kes), 0)       AS max_monthly_wage,
        ROUND(MAX(monthly_wage_kes) 
            - MIN(monthly_wage_kes), 0)       AS wage_spread
    FROM wages
    GROUP BY sector, wage_type
    ORDER BY sector, wage_type
""", conn)

print("Query 3 — Wage Summary by Sector and Type:")
print(q3.to_string(index=False))

Query 3 — Wage Summary by Sector and Type:
 sector wage_type  total_records  avg_monthly_wage  min_monthly_wage  max_monthly_wage  wage_spread
Private   nominal            100           95142.0           22647.0          334084.0     311436.0
Private      real            100           81583.0           20577.0          284790.0     264213.0
 Public   nominal            105           94052.0           39922.0          234697.0     194775.0
 Public      real            105           80777.0           31839.0          188174.0     156335.0


**What we did:**
- We asked the database to take all 410 rows in the wages table and collapse them into 4 summary rows — one for each combination of sector and wage type. Instead of seeing every industry every year, we see the big picture.  
- GROUP BY sector, wage_type is the instruction that does the collapsing. Every row that shares the same sector AND wage type gets grouped together. Then AVG(), MIN(), MAX() calculate summaries across each group.

What the output is telling you — read it column by column:
**total_records**  
Private sector has 100 records per wage type, Public has 105. The small difference is because public sector has a few more industry categories tracked — not a data problem.
**avg_monthly_wage**  
Private nominal average: KES 95,142. Private real average: KES 81,583.
That KES 13,559 gap between nominal and real is inflation — money that appeared on payslips but couldn't buy anything extra.
Public nominal: KES 94,052. Public real: KES 80,777.
Almost identical to private sector. Both sectors eroded at roughly the same rate.
**min_monthly_wage**  
Private sector minimum: KES 22,647 nominal, KES 20,577 real.
That is the lowest-paying industry in the private sector averaged across all years. KES 22,647 per month — KES 271,764 annually. That's agriculture. The people feeding the country.
Public sector minimum: KES 39,922 nominal. Notice public sector's floor is nearly double private sector's floor. Government employment provides a meaningful wage floor that the private sector does not.
**max_monthly_wage**
Private sector maximum: KES 334,084 nominal. That's the electricity and financial services industries at their peak years.
Public sector maximum: KES 234,697 — lower than private sector's peak. High-skill private sector jobs pay more than the best public sector jobs.
**wage_spread**  
This is the most revealing column. Private sector spread: KES 311,436. Public sector spread: KES 194,775.
The private sector has a much wider gap between its best and worst paying industries than the public sector does. In plain terms — the private sector rewards the top much more generously, but also pays the bottom much more poorly. The public sector is more compressed — more equal, but with a lower ceiling.

The one-sentence analytical takeaway:  
*Both sectors experienced nearly identical real wage erosion — but a private sector worker at the bottom of the pay scale started from a much lower base, meaning the same percentage erosion translated into a much smaller absolute amount of money to lose before hitting poverty.*

**Query 4 — Filtering and Ranking with WHERE and ORDER BY**  

Concept: finding the best and worst performers within a specific group
WHERE filters rows before grouping. ORDER BY sorts the final result. LIMIT cuts it to the top N rows. Together these three let you answer "what are the top/bottom X things" — one of the most common questions in any business analysis.

We're going to answer: which industries had the worst real wage decline between 2019 and 2023?

In [7]:
# To find the change between 2019 and 2023 for each industry
# we need to compare two rows for the same industry
# The trick: use two subqueries — one that gets 2019 values, 
# one that gets 2023 values, then JOIN them together

# A subquery is a SELECT inside another SELECT
# Think of it as: create a temporary mini-table, then query that

q4 = pd.read_sql("""
    SELECT
        w2019.sector,
        w2019.industry,
        ROUND(w2019.monthly_wage_kes, 0)  AS real_wage_2019,
        ROUND(w2023.monthly_wage_kes, 0)  AS real_wage_2023,
        ROUND(w2023.monthly_wage_kes 
            - w2019.monthly_wage_kes, 0)  AS change_kes,
        ROUND((w2023.monthly_wage_kes 
            - w2019.monthly_wage_kes) 
            / w2019.monthly_wage_kes 
            * 100, 1)                     AS change_pct
    FROM wages w2019
    JOIN wages w2023
        ON  w2019.industry  = w2023.industry
        AND w2019.sector    = w2023.sector
    WHERE w2019.year        = 2019
        AND w2023.year      = 2023
        AND w2019.wage_type = 'real'
        AND w2023.wage_type = 'real'
    ORDER BY change_pct ASC
    LIMIT 8
""", conn)

print("Query 4 — Industries with Worst Real Wage Decline 2019-2023:")
print(q4.to_string(index=False))

Query 4 — Industries with Worst Real Wage Decline 2019-2023:
 sector                                                            industry  real_wage_2019  real_wage_2023  change_kes  change_pct
 Public       Public administration and defence; compulsory social security         52596.0         38627.0    -13968.0       -26.6
 Public                                                  County governments         68928.0         53100.0    -15828.0       -23.0
 Public Water supply; sewerage, waste management and remediation activities         52564.0         40582.0    -11982.0       -22.8
 Public                 Electricity, gas, steam and air conditioning supply        117301.0         93368.0    -23933.0       -20.4
 Public                                                           Education         57319.0         45650.0    -11668.0       -20.4
 Public                                         Teachers Service Commission         56850.0         45395.0    -11455.0       -20.1
Private        

**What we did in this query:**  
We read the same wages table twice simultaneously — once as w2019 (filtered to 2019 rows) and once as w2023 (filtered to 2023 rows). Then we joined them together by matching industry name and sector, so each row in the result represents one industry with its 2019 value on the left and its 2023 value on the right. That let us calculate the change between the two years in a single query. Without the JOIN trick you'd need two separate queries and manual comparison.

**What the output is telling you:**  
The public sector dominates this list. Seven of the eight worst-performing industries are public sector. This is significant — these are government employees, people who theoretically have job security and structured pay scales, and yet their real wages declined more severely than most private sector workers.
- Public administration and defence: -26.6%
Civil servants and defence personnel lost more than a quarter of their real purchasing power in four years. These are the people running government departments, processing your documents, maintaining public infrastructure. A KES 13,968 monthly decline in real terms is not an abstraction — it is the difference between affording school fees and not.
- County governments: -23.0%
Devolution was meant to bring services and prosperity closer to ordinary Kenyans. Instead county government workers — the nurses, the engineers, the administrators at county level — saw their real wages fall KES 15,828 per month. The largest absolute decline in the entire table.
- Teachers Service Commission: -20.1%
Kenyan teachers lost a fifth of their real purchasing power. This is a policy failure with direct consequences — teacher motivation, absenteeism, and the quality of education all correlate with real compensation. When the people teaching the next generation of Kenyans are getting steadily poorer, the cost shows up decades later.
The one private sector entry — Accommodation and food service: -18.3%
Hotel and restaurant workers were already the lowest paid in the private sector dataset. A further 18.3% real decline pushed their real monthly wage to KES 29,859 — roughly KES 1,000 per day. These workers also face the highest food inflation of any category, meaning they are simultaneously earning less in real terms and paying more for the goods they need most.

The analytical takeaway:
*The public sector real wage collapse is the most underreported story in this dataset. Private sector wage erosion gets attention because it affects formal businesses and shows up in consumer spending data. But civil servants, teachers, county workers, and water utility employees losing 20-27% of their real purchasing power is a public service delivery crisis waiting to unfold — burned-out teachers, underpaid nurses, and demotivated civil servants are the long-term cost of letting public sector real wages erode this severely.*

**Query 5 — HAVING and Consistent Outperformers**  
Concept: filtering groups after aggregation
WHERE filters rows before grouping. HAVING filters groups after grouping. This distinction matters when your condition involves an aggregate function like AVG() or COUNT().
Think of it this way: WHERE says "only include these rows in the calculation." HAVING says "only show me groups where the calculation result meets this condition."  

We're going to answer: which CPI categories beat the national average inflation every single year from 2019 to 2023?

In [10]:
q5 = pd.read_sql("""
    SELECT
        category,
        ROUND(weight_pct, 1)                        AS budget_weight_pct,
        ROUND(AVG(price_change_vs_2019_pct), 1)     AS avg_price_change_pct,
        ROUND(MIN(price_change_vs_2019_pct), 1)     AS min_price_change_pct,
        ROUND(MAX(price_change_vs_2019_pct), 1)     AS max_price_change_pct,
        COUNT(*)                                    AS years_tracked
    FROM cpi_categories
    WHERE category != 'Weighted Average of All Items'
    GROUP BY category, weight_pct
    HAVING AVG(price_change_vs_2019_pct) > (
        SELECT AVG(price_change_vs_2019_pct)
        FROM cpi_categories
        WHERE category = 'Weighted Average of All Items'
    )
    ORDER BY avg_price_change_pct DESC
""", conn)

print("Query 5 — Categories That Consistently Beat the National Average:")
print(q5.to_string(index=False))

Query 5 — Categories That Consistently Beat the National Average:
                        category  budget_weight_pct  avg_price_change_pct  min_price_change_pct  max_price_change_pct  years_tracked
                       Transport                9.6                  22.3                   0.0                  48.2              5
Food and Non-Alcoholic Beverages               32.9                  21.6                   0.0                  46.8              5


**What the query found:**  
Every CPI category was grouped and its average price change across all 5 years was calculated. Then HAVING filtered out every category whose average fell below the national average. Only Transport and Food survived that filter.  

Why only these two matters more than the numbers themselves:  
It means the inflation story is not broadly distributed across the economy. It is concentrated in two categories. Everything else — health, education, ICT, clothing, insurance — ran at or below the national average. The entire inflation burden on Kenyan households is essentially a food and transport problem.

Now read the numbers together:  
Transport averaged +22.3% across 2019-2023, peaking at +48.2% in 2023. Food averaged +21.6%, peaking at +46.8%. Both started at 0.0% in 2019 — the base year — then climbed without reversing once.
The budget weight column is the killer detail:    
Food is 32.9% of the average household budget. Transport is 9.6%. Together that is 42.5% of everything a Kenyan household spends — and those two categories rose more than twice as fast as the national average.
A category like Insurance and Financial Services rose only +4.3% — but it is only 2.2% of the budget. Its low inflation barely helps anyone. The categories that are cheap to keep up with are small. The categories that are expensive to keep up with are enormous.

The one sentence this query proves:  
*Kenya's inflation crisis between 2019 and 2023 was not a broad-based price problem — it was a food and fuel crisis that hit hardest precisely because those two things dominate the budgets of the people with the least room to absorb the shock.*

**Query 6 — Window Functions with LAG**  
Concept: comparing each row to the previous row
This is where SQL starts to feel powerful. A window function performs a calculation across a set of rows that are related to the current row — without collapsing them into a group like GROUP BY does. Every row stays in the result, but each row gains a new column calculated from its neighbours.  
LAG(column, 1) looks back one row and returns that value. So if you're on the 2022 row, LAG(inflation_pct, 1) returns the 2021 value. This lets you calculate year-on-year changes directly in SQL.  

In [11]:
q6 = pd.read_sql("""
    SELECT
        year,
        ROUND(inflation_pct, 2)                     AS inflation_pct,
        ROUND(govt_debt_pct_gdp, 1)                 AS debt_pct_gdp,
        ROUND(national_savings_pct_gdp, 1)          AS savings_pct_gdp,
        ROUND(govt_debt_pct_gdp 
            - LAG(govt_debt_pct_gdp, 1) 
            OVER (ORDER BY year), 1)                AS debt_change_vs_prev_yr,
        ROUND(national_savings_pct_gdp
            - LAG(national_savings_pct_gdp, 1)
            OVER (ORDER BY year), 1)                AS savings_change_vs_prev_yr
    FROM macro
    ORDER BY year
""", conn)

print("Query 6 — Debt and Savings Shifts Year on Year:")
print(q6.to_string(index=False))

Query 6 — Debt and Savings Shifts Year on Year:
 year  inflation_pct  debt_pct_gdp  savings_pct_gdp  debt_change_vs_prev_yr  savings_change_vs_prev_yr
 2018           4.69          56.4             14.0                     NaN                        NaN
 2019           5.24          59.1             14.1                     2.6                        0.1
 2020           5.41          68.0             14.9                     8.9                        0.8
 2021           6.11          68.2             15.2                     0.3                        0.3
 2022           7.66          67.8             14.0                    -0.4                       -1.1
 2023           7.67          73.1             12.4                     5.3                       -1.6
 2024           4.49          69.9             12.6                    -3.2                        0.2


## Query 6 — What the Output Tells Us

### What we built
A year-on-year change table using the LAG() window function. Instead of 
just showing debt and savings levels, we now see how much each moved 
compared to the previous year. This reveals momentum — which years saw 
the sharpest deterioration and which saw stabilisation.

The NaN in 2018 is expected and correct — there is no previous year in 
the dataset to compare against, so LAG() returns NULL for that row.

### Reading the table as a policy story

**2019 — Manageable drift**
Debt rose 2.6 percentage points. Savings barely moved (+0.1pp). 
Government was borrowing modestly and households were holding steady. 
Pre-shock stability.

**2020 — The COVID borrowing surge**
The single largest debt jump in the dataset: +8.9pp in one year, 
pushing debt from 59% to 68% of GDP. Government borrowed massively 
to fund emergency responses and cover revenue shortfalls as the 
economy contracted. Household savings actually rose +0.8pp — not 
because incomes improved, but because lockdowns forced reduced spending.

**2021 — Brief stabilisation**
Debt barely moved (+0.3pp) as the economy rebounded strongly. 
Savings ticked up to 15.2% — the highest in the entire dataset. 
The one year where both indicators moved in the right direction 
simultaneously.

**2022 — The turn**
Debt marginally fell -0.4pp but savings dropped -1.1pp. Inflation 
hit 7.66% and households started drawing down savings to cover rising 
costs. This is the first hard signal that inflation was eating 
household financial resilience.

**2023 — The crisis year**
The worst combination in the dataset. Debt surged +5.3pp to 73.1% 
of GDP — the highest on record — while savings fell -1.6pp to 12.4%, 
the lowest since 2018. Government borrowing more while households 
save less. Both trends pulling in the wrong direction simultaneously.

**2024 — Early stabilisation**
Debt fell -3.2pp as inflation eased. Savings ticked up marginally 
+0.2pp. Some relief — but savings at 12.6% remains well below the 
2020–2021 levels. The financial cushion built during COVID has been 
spent down by three years of high inflation.

### The connection to the broader analysis
The 2022–2023 savings collapse runs exactly parallel to the real wage 
decline in Chart 4 and the high-pressure inflation in Chart 2. When 
real wages fall and prices rise simultaneously, households do not 
reduce consumption proportionally — they draw down savings first.

In a country where the majority of households have no formal savings 
buffer to begin with, a savings rate decline from 15.2% to 12.4% 
between 2021 and 2023 has consequences that compound long after 
inflation eases. Reduced savings means reduced resilience to the 
next shock — whatever form it takes.

### What LAG() taught us about SQL
This query demonstrates why window functions matter. Without LAG() 
you would need a self-JOIN (like Query 4) for every single year 
comparison — seven separate joins for seven years. LAG() does it 
in one clean column. For time series data — which is what most 
economic and business data is — window functions are not optional. 
They are the standard tool.

**Query 7 — CTE (Common Table Expression) with WITH**  
Concept: naming a subquery so you can reuse it  
A CTE lets you define a temporary named result set at the top of your query using WITH. Think of it as creating a mini-table that only exists for the duration of that one query. It makes complex queries readable — instead of nesting subqueries inside each other like Russian dolls, you name each step and build on it cleanly.
This is one of the most common patterns in professional SQL work. Any time you find yourself writing a subquery inside a subquery, a CTE is the cleaner solution.

The question we're answering: for each year, how much of the nominal wage increase was real vs illusory — and how does that map to the household cost burden of food and transport specifically?

In [12]:
q7 = pd.read_sql("""
    -- Step 1: define the CTE
    -- WITH gives it a name — we call it 'wage_cpi'
    -- Everything inside the brackets is a regular SELECT
    WITH wage_cpi AS (
        SELECT
            m.year,
            -- Wages — only available 2019-2023
            ROUND(m.avg_nominal_monthly_wage, 0)    AS nominal_wage,
            ROUND(m.avg_real_monthly_wage, 0)       AS real_wage,
            ROUND(m.avg_nominal_monthly_wage 
                - m.avg_real_monthly_wage, 0)       AS wage_illusion_kes,

            -- CPI categories from same table
            ROUND(m.cpi_food, 1)                    AS cpi_food,
            ROUND(m.cpi_transport, 1)               AS cpi_transport,
            ROUND(m.cpi_overall, 1)                 AS cpi_overall,

            -- Inflation burden: food+transport weighted share of total inflation
            -- Food weight 32.9%, transport weight 9.6% = 42.5% combined
            ROUND(
                (m.cpi_food    * 0.329 + 
                 m.cpi_transport * 0.096) / 
                (0.329 + 0.096), 1
            )                                       AS weighted_burden_index
        FROM macro m
        WHERE m.avg_nominal_monthly_wage IS NOT NULL
    )

    -- Step 2: query the CTE like a normal table
    -- We can now reference 'wage_cpi' by name
    SELECT
        year,
        nominal_wage,
        real_wage,
        wage_illusion_kes,
        cpi_food,
        cpi_transport,
        cpi_overall,
        weighted_burden_index,
        -- How much more than overall CPI is the food+transport burden?
        ROUND(weighted_burden_index - cpi_overall, 1) AS burden_above_average
    FROM wage_cpi
    ORDER BY year
""", conn)

print("Query 7 — Wage Illusion vs Household Cost Burden:")
print(q7.to_string(index=False))

Query 7 — Wage Illusion vs Household Cost Burden:
 year  nominal_wage  real_wage  wage_illusion_kes  cpi_food  cpi_transport  cpi_overall  weighted_burden_index  burden_above_average
 2019       87772.0    84788.0             2984.0     107.0          102.4        103.2                  106.0                   2.8
 2020       92033.0    85007.0             7026.0     116.7          111.5        108.7                  115.5                   6.8
 2021       93791.0    81480.0            12311.0     126.7          125.2        115.3                  126.3                  11.0
 2022       98953.0    79660.0            19293.0     143.3          135.3        124.2                  141.5                  17.3
 2023      103160.0    76979.0            26181.0     157.1          151.7        133.7                  155.9                  22.2


**What Query 7 did:**  
We used a CTE to build a mini-table called wage_cpi that combined wage data and CPI data from the macro table in one place. Then we queried that mini-table to calculate two derived measures that don't exist anywhere in the raw data — the wage illusion in KES, and how much harder food and transport inflation hit compared to the overall average.
The CTE made this readable. Without it you'd have nested subqueries three levels deep. With it, you define the building blocks first, then ask your question on top of them.

What the output is telling you — column by column:
wage_illusion_kes — this is the money that appeared on payslips but couldn't buy anything extra. In 2019 it was KES 2,984 per month — small, almost acceptable. By 2023 it had grown to KES 26,181. That means in 2023, KES 26,181 of every worker's monthly wage was effectively phantom income — it existed numerically but was completely consumed by inflation before it could improve anyone's standard of living.
To put that in perspective: KES 26,181 per month is more than the entire monthly wage of an agriculture worker in 2019. The illusion alone was worth more than a full salary at the bottom of the pay scale.
cpi_food and cpi_transport — both indices climbing every single year without a single reversal. Food went from 107.0 in 2019 to 157.1 in 2023. Transport from 102.4 to 151.7. Both using February 2016 as base 100 — meaning by 2023, food cost 57% more than 2016 and transport 52% more.
cpi_overall — the weighted average across all 13 categories. Went from 103.2 to 133.7. This is the number the government reports as the official inflation index.
weighted_burden_index — this is the number we calculated ourselves by combining food and transport at their actual budget weights. It went from 106.0 to 155.9. Notice it is consistently higher than cpi_overall every single year — and the gap keeps growing.
burden_above_average — this is the analytical punchline of the entire query. It is the difference between the weighted burden index and the overall CPI. Read it year by year:
YearBurden Above Average2019+2.8 points2020+6.8 points2021+11.0 points2022+17.3 points2023+22.2 points
Every year the gap widens. By 2023 the food and transport burden index was running 22.2 index points above the overall average. That is not a rounding error. That is a structural and accelerating divergence between what official inflation statistics report and what households whose budgets are dominated by food and transport actually experienced.

The single most important sentence this query proves:
The official inflation figure understates the real cost burden on low and middle income Kenyan households by a margin that grew from 2.8 points in 2019 to 22.2 points in 2023 — because official inflation averages across all categories equally, while poor households cannot average their way out of food and transport costs that dominate their budgets.

**Query 8 — The Executive Summary**  
Concept: pulling everything together in one query  
This is the most important query in the project. It does not introduce a new SQL concept — it combines everything you have learned: SELECT, ROUND, CASE, WITH, subqueries, and calculated columns. The goal is one clean table that tells the entire story of this project in a single glance.

This is the query a CFO, a policy analyst, or a hiring manager runs when they want the headline numbers without reading the full analysis. In a real job this is what gets pasted into a board presentation.

In [13]:
q8 = pd.read_sql("""
    WITH summary AS (
        SELECT
            year,

            -- ── MACRO THEME ───────────────────────────────────
            ROUND(inflation_pct, 1)                     AS inflation_pct,
            ROUND(real_growth_vs_2018_pct, 1)           AS real_income_growth_pct,
            ROUND(purchasing_power_of_1000kes, 0)       AS kes_1000_worth,

            -- ── WAGES THEME ───────────────────────────────────
            ROUND(avg_nominal_monthly_wage, 0)          AS nominal_wage,
            ROUND(avg_real_monthly_wage, 0)             AS real_wage,
            ROUND(real_wage_change_vs_2019_pct, 1)      AS real_wage_change_pct,

            -- ── COST BURDEN THEME ─────────────────────────────
            ROUND(cpi_food, 1)                          AS cpi_food,
            ROUND(cpi_transport, 1)                     AS cpi_transport,
            ROUND(petrol_kes_litre, 0)                  AS petrol_kes_litre,

            -- ── INEQUALITY THEME ──────────────────────────────
            ROUND(inflation_lower_income, 1)            AS inflation_lower,
            ROUND(inflation_upper_income, 1)            AS inflation_upper,
            ROUND(inflation_gap_lower_vs_upper, 1)      AS inequality_gap_pp,

            -- ── FISCAL THEME ──────────────────────────────────
            ROUND(govt_debt_pct_gdp, 1)                 AS govt_debt_pct_gdp,
            ROUND(national_savings_pct_gdp, 1)          AS savings_pct_gdp,

            -- ── OVERALL HEALTH SCORE ──────────────────────────
            -- Score from 0-5: one point deducted for each bad signal
            -- This is a simple composite — not a formal index
            5
            - CASE WHEN inflation_pct >= 7.0        THEN 1 ELSE 0 END
            - CASE WHEN real_growth_vs_2018_pct <= 0 THEN 1 ELSE 0 END
            - CASE WHEN real_wage_change_vs_2019_pct < -5 THEN 1 ELSE 0 END
            - CASE WHEN govt_debt_pct_gdp >= 70     THEN 1 ELSE 0 END
            - CASE WHEN national_savings_pct_gdp < 13 THEN 1 ELSE 0 END
                                                        AS household_health_score
        FROM macro
    )

    SELECT
        year,
        inflation_pct,
        real_income_growth_pct,
        kes_1000_worth,
        nominal_wage,
        real_wage,
        real_wage_change_pct,
        cpi_food,
        cpi_transport,
        petrol_kes_litre,
        inflation_lower,
        inflation_upper,
        inequality_gap_pp,
        govt_debt_pct_gdp,
        savings_pct_gdp,
        household_health_score,
        CASE household_health_score
            WHEN 5 THEN 'Healthy'
            WHEN 4 THEN 'Manageable'
            WHEN 3 THEN 'Under pressure'
            WHEN 2 THEN 'Stressed'
            WHEN 1 THEN 'Critical'
            ELSE        'Crisis'
        END                                             AS household_status
    FROM summary
    ORDER BY year
""", conn)

print("Query 8 — Executive Summary: The Real Cost of Being Kenyan")
print()

# Print transposed — rows become columns for readability
# This format works better for a wide table with many indicators
print(q8.set_index('year').T.to_string())

Query 8 — Executive Summary: The Real Cost of Being Kenyan

year                          2018     2019        2020     2021            2022      2023        2024
inflation_pct                  4.7      5.2         5.4      6.1             7.7       7.7         4.5
real_income_growth_pct         0.0      2.1        -0.6      3.2             5.5       7.0         8.4
kes_1000_worth              1000.0    950.0       901.0    850.0           789.0     733.0       701.0
nominal_wage                   NaN  87772.0     92033.0  93791.0         98953.0  103160.0         NaN
real_wage                      NaN  84788.0     85007.0  81480.0         79660.0   76979.0         NaN
real_wage_change_pct           NaN      0.0         0.3     -3.9            -6.0      -9.2         NaN
cpi_food                       NaN    107.0       116.7    126.7           143.3     157.1         NaN
cpi_transport                  NaN    102.4       111.5    125.2           135.3     151.7         NaN
petrol_kes_li

## Query 8 — Executive Summary: The Real Cost of Being Kenyan

### What this query built
This is the capstone query of the entire SQL layer. It combines every 
theme from the previous 7 queries — macro, wages, cost burden, 
inequality, and fiscal — into a single transposed table that tells 
the full story of Kenya's economic reality from 2018 to 2024 in one 
glance.

The two new elements introduced here are:

**Composite scoring** — five CASE statements each return 1 (bad signal 
present) or 0 (signal absent). Subtracting all five from 5 produces a 
score between 0 and 5. This is a standard SQL pattern for building 
simple indicators without leaving the database.

**Transposed output** — the table is flipped so years become column 
headers and indicators become rows. For a wide summary table this 
orientation is far more readable than the standard format.

---

### The scoring criteria
Each year loses one point for each of the following bad signals:

| Signal | Threshold | Meaning |
|--------|-----------|---------|
| High inflation | ≥ 7.0% | Prices rising faster than wages can absorb |
| Negative real growth | ≤ 0% vs 2018 | Economy shrinking in real terms |
| Severe wage erosion | < -5% vs 2019 | Workers meaningfully poorer |
| Dangerous debt | ≥ 70% of GDP | Government borrowing at risk levels |
| Low savings | < 13% of GDP | Households running out of financial buffer |

A score of 5 means none of these signals are present — Healthy.
A score of 1 means four of the five signals are firing simultaneously — Critical.

---

### Reading the results year by year

**2018